# Deep Learning PR 2 — Adult Income Classification
**Red & White Skill Education | Deep Learning | PR 2**

**Objective:** Build and compare Multi-Layer Perceptron (MLP) models for predicting whether annual income exceeds $50K using the Adult Census Income dataset.

This notebook follows the supplied project brief and covers data cleaning/EDA, MLP architecture, activation functions, weight initialization, loss functions, batch normalization, and optimizers.

> **Before running:** keep `adult.csv.zip` (or the extracted `adult.csv`) in the same folder as this notebook. The experiments train many neural networks and can take a while. Results are calculated when you run the notebook; no performance values are fabricated in advance.

## 0. Imports and reproducibility

In [ ]:
import os, zipfile, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay
)
from sklearn.utils.class_weight import compute_class_weight
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

## Task 1 — Data Loading, Cleaning & Exploratory Data Analysis
**Objective:** Inspect and clean the Adult dataset, explore class imbalance and feature relationships, then prepare leakage-aware training and test data.

**Expected output:** Dataset shapes, missing-value summary, four EDA plots, and stratified train/test shapes.

In [ ]:
# Load either adult.csv or the provided adult.csv.zip.
if os.path.exists("adult.csv"):
    data_path = "adult.csv"
elif os.path.exists("adult.csv.zip"):
    with zipfile.ZipFile("adult.csv.zip") as z:
        csv_name = next((n for n in z.namelist() if n.lower().endswith(".csv")), None)
        if csv_name is None:
            raise FileNotFoundError("No CSV file found inside adult.csv.zip")
        with z.open(csv_name) as f:
            df = pd.read_csv(f, skipinitialspace=True)
else:
    raise FileNotFoundError("Place adult.csv or adult.csv.zip beside this notebook.")

# If loaded directly above, df already exists; otherwise read CSV.
if "df" not in globals():
    df = pd.read_csv(data_path, skipinitialspace=True)

print("Original shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))

In [ ]:
# Normalize whitespace in all string columns and convert '?' to missing values.
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()
df.replace("?", np.nan, inplace=True)

print("Missing values by column:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Rows before dropping missing values:", len(df))
df.dropna(inplace=True)
print("Shape after dropping missing rows:", df.shape)

**Why drop rows with missing values?** The project brief specifies dropping rows containing missing values. The affected fields are categorical and contain multiple categories; mode-imputing them could over-represent the most common category. Row deletion also has a trade-off: it reduces the dataset, so the resulting sample should be interpreted with that limitation in mind.

In [ ]:
# Remove census sampling weight and redundant text education feature.
drop_cols = [c for c in ["fnlwgt", "education"] if c in df.columns]
df.drop(columns=drop_cols, inplace=True)

if "income" not in df.columns:
    raise ValueError("Expected target column named 'income' in adult.csv.")

# Encode target: <=50K -> 0, >50K -> 1
df["income"] = (df["income"].astype(str).str.strip() == ">50K").astype(int)
print("Cleaned shape:", df.shape)
display(df["income"].value_counts().rename(index={0:"<=50K (0)", 1:">50K (1)"}).to_frame("count"))
display((df["income"].value_counts(normalize=True)*100).rename(index={0:"<=50K (0)", 1:">50K (1)"}).round(2).to_frame("percent"))

In [ ]:
plt.figure(figsize=(6,4))
ax = sns.countplot(data=df, x="income")
ax.set_xticks([0,1], labels=["<=50K (0)", ">50K (1)"])
ax.set_title("Income Class Distribution")
ax.set_xlabel("Annual income class")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.show()

**Class imbalance:** The dataset is expected to contain substantially more `<=50K` than `>50K` records. A classifier that always predicts the majority class can have deceptively high accuracy. Therefore, precision, recall, F1-score for class 1, and ROC-AUC are tracked in addition to accuracy.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.histplot(data=df, x="age", hue="income", bins=30, stat="count",
             common_norm=False, element="step", ax=axes[0,0])
axes[0,0].set_title("Age Distribution by Income")
sns.boxplot(data=df, x="income", y="hours.per.week", ax=axes[0,1])
axes[0,1].set_xticks([0,1], labels=["<=50K", ">50K"])
axes[0,1].set_title("Hours per Week by Income")
sns.countplot(data=df, x="education.num", hue="income", ax=axes[1,0])
axes[1,0].set_title("Education Level by Income")
axes[1,0].legend(title="Income", labels=["<=50K", ">50K"])
numeric_for_corr = df.select_dtypes(include=np.number)
sns.heatmap(numeric_for_corr.corr(), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=axes[1,1])
axes[1,1].set_title("Numeric Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

**EDA interpretation prompts (complete after inspecting your plots):**
- Compare the age distributions for the two income classes.
- Compare median and spread of `hours.per.week` between classes.
- Check whether higher `education.num` values appear more often in the `>50K` class.
- Use the correlation heatmap to identify numeric associations; correlation does not establish causation.

In [ ]:
# One-hot encode categorical features. Keep the binary target separate.
X_raw = df.drop(columns=["income"])
y = df["income"].astype("int32")

X = pd.get_dummies(X_raw, drop_first=True, dtype=float)
print("Feature count after one-hot encoding:", X.shape[1])

# Stratified split first, then fit scaler on training data only to avoid test leakage.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
numeric_cols = [c for c in ["age", "education.num", "capital.gain",
                            "capital.loss", "hours.per.week"] if c in X_train_raw.columns]
scaler = StandardScaler()
X_train = X_train_raw.copy()
X_test = X_test_raw.copy()
X_train.loc[:, numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test.loc[:, numeric_cols] = scaler.transform(X_test[numeric_cols])

# Hold out a validation set from training data for comparable per-epoch diagnostics.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=SEED, stratify=y_train
)
# TensorFlow Dense layers work reliably with float32 arrays.
X_fit = X_fit.astype("float32")
X_val = X_val.astype("float32")
X_train = X_train.astype("float32")
X_test = X_test.astype("float32")
y_fit = y_fit.astype("float32")
y_val = y_val.astype("float32")
y_train = y_train.astype("int32")
y_test = y_test.astype("int32")

print("Fit:", X_fit.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
print("Class-1 proportions — fit/validation/test:",
      round(float(y_fit.mean()), 3), round(float(y_val.mean()), 3), round(float(y_test.mean()), 3))

**Preprocessing note:** Numeric columns are standardized, while one-hot encoded binary indicator columns are left as 0/1. The scaler is fitted only on the training split to prevent information from the test set leaking into preprocessing.

## Shared utilities — model builder, training, and evaluation

In [ ]:
def build_ann(input_dim, hidden_units=(128, 64), activation="relu",
              initializer="glorot_uniform", use_batch_norm=False,
              optimizer="adam", loss="binary_crossentropy"):
    """Build a configurable MLP for binary income classification."""
    model = tf.keras.Sequential(name="adult_income_ann")
    model.add(tf.keras.layers.Input(shape=(input_dim,)))
    for units in hidden_units:
        model.add(tf.keras.layers.Dense(units, kernel_initializer=initializer))
        if use_batch_norm:
            model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Activation(activation))
    model.add(tf.keras.layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer=optimizer, loss=loss, metrics=["accuracy"])
    return model

def make_f1_callback(Xv=X_val, yv=y_val):
    class ValidationF1(tf.keras.callbacks.Callback):
        def __init__(self):
            super().__init__()
            self.val_f1 = []
        def on_epoch_end(self, epoch, logs=None):
            probs = self.model.predict(Xv, verbose=0).ravel()
            pred = (probs >= 0.5).astype(int)
            self.val_f1.append(f1_score(yv, pred, zero_division=0))
    return ValidationF1()

def train_model(model, epochs=50, class_weight=None, verbose=0):
    f1_cb = make_f1_callback()
    history = model.fit(
        X_fit, y_fit, validation_data=(X_val, y_val),
        epochs=epochs, batch_size=64, class_weight=class_weight,
        callbacks=[f1_cb], verbose=verbose
    )
    history.val_f1_class1 = f1_cb.val_f1
    return history

def evaluate_model(model, name="Model", threshold=0.5):
    prob = model.predict(X_test, verbose=0).ravel()
    pred = (prob >= threshold).astype(int)
    result = {
        "Model": name,
        "Test Acc": accuracy_score(y_test, pred),
        "Prec(1)": precision_score(y_test, pred, zero_division=0),
        "Recall(1)": recall_score(y_test, pred, zero_division=0),
        "F1(1)": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    }
    return result, prob, pred

results = []
histories = {}
models = {}

## Task 2 — MLP Architecture: Baseline ANN
**Objective:** Establish a reusable MLP and evaluate a baseline model.

**Expected output:** Model summary, training curves, classification report, and confusion matrix.

In [ ]:
model_base = build_ann(input_dim=X_fit.shape[1])
model_base.summary()

# Parameter-count check for Dense(128) -> Dense(64) -> Dense(1)
input_dim = X_fit.shape[1]
manual_params = (input_dim*128 + 128) + (128*64 + 64) + (64*1 + 1)
print("Manual parameter count (without BatchNorm):", manual_params)

**Why sigmoid rather than softmax at the output?** This is binary classification. A single sigmoid neuron outputs a probability between 0 and 1 for class `>50K`. A two-neuron softmax output is also possible, but it is unnecessary for this binary setup.

**Why an MLP for tabular data?** A fully connected network accepts scaled numeric and one-hot encoded categorical features and can learn nonlinear combinations of features. It is a reasonable neural-network baseline for structured demographic and employment data.

In [ ]:
histories["Baseline"] = train_model(model_base, epochs=50, verbose=1)
models["Baseline"] = model_base

fig, axes = plt.subplots(1, 2, figsize=(13,4))
axes[0].plot(histories["Baseline"].history["loss"], label="Train")
axes[0].plot(histories["Baseline"].history["val_loss"], label="Validation")
axes[0].set_title("Baseline ANN — Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(histories["Baseline"].history["accuracy"], label="Train")
axes[1].plot(histories["Baseline"].history["val_accuracy"], label="Validation")
axes[1].set_title("Baseline ANN — Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

base_result, base_prob, base_pred = evaluate_model(model_base, "Baseline")
results.append(base_result)
print(classification_report(y_test, base_pred, target_names=["<=50K (0)", ">50K (1)"], zero_division=0))
sns.heatmap(confusion_matrix(y_test, base_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=["<=50K", ">50K"], yticklabels=["<=50K", ">50K"])
plt.title("Baseline ANN — Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.tight_layout(); plt.show()

## Task 3 — Activation Functions
**Objective:** Compare ReLU, tanh, sigmoid, and ELU in the hidden layers and inspect ReLU activations and sigmoid gradients.

**Expected output:** 2×2 accuracy comparison, dead-neuron estimate, gradient plot, and activation summary.

In [ ]:
activation_histories, activation_models, activation_scores = {}, {}, {}
for act in ["relu", "tanh", "sigmoid", "elu"]:
    print("Training activation:", act)
    m = build_ann(input_dim=X_fit.shape[1], activation=act)
    h = train_model(m, epochs=50, verbose=0)
    activation_models[act], activation_histories[act] = m, h
    prob = m.predict(X_val, verbose=0).ravel()
    activation_scores[act] = {
        "Val Accuracy": accuracy_score(y_val, prob >= 0.5),
        "Val F1(1)": f1_score(y_val, prob >= 0.5, zero_division=0)
    }
    models["Activation: " + act] = m
    results.append(evaluate_model(m, "Activation: " + act)[0])

fig, axes = plt.subplots(2, 2, figsize=(13,9))
for ax, act in zip(axes.ravel(), ["relu", "tanh", "sigmoid", "elu"]):
    h = activation_histories[act]
    ax.plot(h.history["accuracy"], label="Train")
    ax.plot(h.history["val_accuracy"], label="Validation")
    ax.axhline(0.75, linestyle="--", color="gray", label="75% reference")
    ax.set_title(f"{act.upper()} | val acc={activation_scores[act]['Val Accuracy']:.3f}, "
                 f"F1(1)={activation_scores[act]['Val F1(1)']:.3f}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ReLU dead-neuron diagnostic: proportion of first hidden-layer activations exactly zero.
relu_model = activation_models["relu"]
first_hidden = next(layer for layer in relu_model.layers if isinstance(layer, tf.keras.layers.Dense))
intermediate = tf.keras.Model(inputs=relu_model.input, outputs=first_hidden.output)
activations = intermediate.predict(X_test[:500], verbose=0)
dead_fraction = float(np.mean(activations == 0))
print(f"Fraction of zero first-layer activations on 500 test rows: {dead_fraction:.4f}")
plt.hist(np.mean(activations, axis=0), bins=20, edgecolor="black")
plt.title("ReLU First-Layer Activation Distribution")
plt.xlabel("Mean activation per neuron"); plt.ylabel("Number of neurons")
plt.tight_layout(); plt.show()

In [ ]:
# Gradient magnitude by layer for the sigmoid-hidden model.
sigmoid_model = activation_models["sigmoid"]
xb = tf.convert_to_tensor(X_fit[:256], dtype=tf.float32)
yb = tf.reshape(tf.convert_to_tensor(y_fit[:256], dtype=tf.float32), (-1, 1))
with tf.GradientTape() as tape:
    pred = sigmoid_model(xb, training=True)
    loss_value = tf.keras.losses.binary_crossentropy(yb, pred)
    loss_value = tf.reduce_mean(loss_value)
grads = tape.gradient(loss_value, sigmoid_model.trainable_weights)
layer_gradients = {}
for layer in sigmoid_model.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        vals = []
        for weight, grad in zip(sigmoid_model.trainable_weights, grads):
            if grad is not None and (weight is layer.kernel or weight is layer.bias):
                vals.append(tf.reduce_mean(tf.abs(grad)).numpy())
        if vals:
            layer_gradients[layer.name] = float(np.mean(vals))
plt.barh(list(layer_gradients.keys()), list(layer_gradients.values()))
plt.xlabel("Mean absolute gradient"); plt.title("Gradient Magnitude by Dense Layer — Sigmoid Hidden Activations")
plt.tight_layout(); plt.show()
display(pd.DataFrame([
    {"Activation": a, **activation_scores[a]} for a in activation_scores
]).sort_values("Val F1(1)", ascending=False))

**Activation interpretation:** ReLU is computationally simple but can produce zero outputs for negative inputs (the “dead neuron” concern). Tanh is zero-centred but can saturate. Sigmoid hidden units can saturate and contribute to vanishing gradients because its derivative is at most 0.25. ELU permits small negative outputs and may reduce dead-neuron behaviour. Choose the best activation based on the measured validation/test metrics in this run, not on a predetermined assumption.

## Task 4 — Weight Initialization Techniques
**Objective:** Compare Glorot Uniform, Glorot Normal, He Uniform, He Normal, and zero initialization.

**Expected output:** Convergence curves, zero-initialization failure demonstration, and before/after weight distributions.

In [ ]:
initializers = ["glorot_uniform", "glorot_normal", "he_uniform", "he_normal", "zeros"]
init_models, init_histories, init_weights_before = {}, {}, {}
for init in initializers:
    print("Training initializer:", init)
    m = build_ann(input_dim=X_fit.shape[1], activation="relu", initializer=init)
    dense_layers = [l for l in m.layers if isinstance(l, tf.keras.layers.Dense)]
    init_weights_before[init] = dense_layers[0].get_weights()[0].copy()
    h = train_model(m, epochs=50, verbose=0)
    init_models[init], init_histories[init] = m, h
    models["Init: " + init] = m
    results.append(evaluate_model(m, "Init: " + init)[0])

plt.figure(figsize=(12,6))
for init, h in init_histories.items():
    plt.plot(h.history["val_accuracy"], label=init)
    crossed = np.where(np.array(h.history["val_accuracy"]) >= 0.82)[0]
    if len(crossed):
        plt.axvline(crossed[0], linestyle=":", alpha=0.35)
plt.axhline(0.84, linestyle="--", color="black", label="84% target")
plt.title("Weight Initialization — Convergence Comparison")
plt.xlabel("Epoch"); plt.ylabel("Validation accuracy"); plt.legend()
plt.tight_layout(); plt.show()

**Why initialization matters:** All-zero weights make neurons within a layer behave identically, so they receive identical gradients and fail to learn distinct features. Glorot/Xavier initialization uses a variance related to `2 / (fan_in + fan_out)` and is often used with tanh-like activations. He initialization uses variance `2 / fan_in`, which is designed for ReLU-family activations.

In [ ]:
zero_h = init_histories["zeros"]
fig, ax1 = plt.subplots(figsize=(10,4))
ax1.plot(zero_h.history["loss"], label="Training loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Training loss")
ax2 = ax1.twinx()
ax2.plot(zero_h.history["val_accuracy"], color="tab:orange", label="Validation accuracy")
ax2.set_ylabel("Validation accuracy")
plt.title("Zero Initialization — Failure Demonstration")
fig.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12,8))
for ax, init, phase in [
    (axes[0,0], "he_normal", "before"),
    (axes[0,1], "he_normal", "after"),
    (axes[1,0], "glorot_uniform", "before"),
    (axes[1,1], "glorot_uniform", "after")
]:
    weights = init_weights_before[init] if phase == "before" else [
        l for l in init_models[init].layers if isinstance(l, tf.keras.layers.Dense)
    ][0].get_weights()[0]
    ax.hist(np.asarray(weights).ravel(), bins=40, edgecolor="black")
    ax.set_title(f"{init} — {phase} training")
    ax.set_xlabel("Weight value"); ax.set_ylabel("Frequency")
plt.suptitle("Weight Distributions — Initial vs Trained")
plt.tight_layout(); plt.show()

## Task 5 — Loss Functions
**Objective:** Compare binary cross-entropy (BCE), mean squared error (MSE), weighted BCE, and focal loss on minority-class performance.

**Expected output:** Validation F1 curves, precision/recall/F1 comparison, and a loss summary.

**Binary cross-entropy (BCE):** `L = -[y log(p) + (1-y) log(1-p)]`, where `p` is the sigmoid probability and `y` is the binary target. BCE is a natural loss for Bernoulli classification and strongly penalizes confident incorrect predictions. MSE treats the probability more like a regression output and can provide weaker gradients when a sigmoid saturates.

**Weighted BCE:** gives more importance to under-represented classes through `class_weight`. **Focal loss:** down-weights easy examples and focuses learning on harder examples; its effect must be verified on validation metrics.

In [ ]:
# MSE loss experiment, retaining sigmoid output.
model_mse = build_ann(input_dim=X_fit.shape[1], activation="relu",
                      initializer="he_normal", loss="mean_squared_error")
history_mse = train_model(model_mse, epochs=50, verbose=0)
models["MSE"] = model_mse
results.append(evaluate_model(model_mse, "MSE")[0])

# Balanced class weights for weighted BCE.
class_values = np.array([0, 1])
cw = compute_class_weight(class_weight="balanced", classes=class_values, y=np.asarray(y_fit, dtype=int))
class_weight_dict = {0: float(cw[0]), 1: float(cw[1])}
print("Class weights:", class_weight_dict)
model_wbce = build_ann(input_dim=X_fit.shape[1], activation="relu",
                       initializer="he_normal", loss="binary_crossentropy")
history_wbce = train_model(model_wbce, epochs=50, class_weight=class_weight_dict, verbose=0)
models["Weighted BCE"] = model_wbce
results.append(evaluate_model(model_wbce, "Weighted BCE")[0])

def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, (-1, 1)), tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce = -y_true * tf.math.log(y_pred) - (1.0-y_true) * tf.math.log(1.0-y_pred)
        weight = alpha*y_true*tf.pow(1.0-y_pred, gamma) + (1.0-alpha)*(1.0-y_true)*tf.pow(y_pred, gamma)
        return tf.reduce_mean(weight * bce)
    return loss

model_focal = build_ann(input_dim=X_fit.shape[1], activation="relu",
                        initializer="he_normal", loss=focal_loss(gamma=2.0, alpha=0.25))
history_focal = train_model(model_focal, epochs=50, verbose=0)
models["Focal Loss"] = model_focal
results.append(evaluate_model(model_focal, "Focal Loss")[0])

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history_mse.val_f1_class1, label="MSE")
plt.plot(histories["Baseline"].val_f1_class1, label="Baseline BCE")
plt.title("Validation F1-score (Class 1): BCE vs MSE")
plt.xlabel("Epoch"); plt.ylabel("F1-score"); plt.legend(); plt.tight_layout(); plt.show()

metric_rows = []
for label, model in [("Baseline BCE", model_base), ("Weighted BCE", model_wbce), ("Focal Loss", model_focal)]:
    r, _, _ = evaluate_model(model, label)
    metric_rows.append({"Model": label, "Precision(1)": r["Prec(1)"],
                        "Recall(1)": r["Recall(1)"], "F1(1)": r["F1(1)"]})
display(pd.DataFrame(metric_rows).set_index("Model").plot(kind="bar", figsize=(10,5), ylim=(0,1)).figure)
plt.title("Class-1 Metrics: Baseline vs Weighted BCE vs Focal Loss")
plt.tight_layout(); plt.show()

**Loss recommendation:** Use the measured class-1 F1, recall, precision, and ROC-AUC to choose the loss. Weighted BCE may improve recall for the minority class but can reduce precision. Focal loss is not guaranteed to outperform weighted BCE on every tabular dataset.

## Task 6 — Batch Normalization
**Objective:** Compare a baseline MLP with batch normalization, test BN placement, and inspect learned gamma/beta parameters.

**Expected output:** Training dynamics, test metrics, BN position comparison, and parameter plots.

In [ ]:
model_bn = build_ann(input_dim=X_fit.shape[1], activation="relu",
                     initializer="he_normal", use_batch_norm=True)
model_bn.summary()
history_bn = train_model(model_bn, epochs=50, verbose=0)
models["BatchNorm"] = model_bn
results.append(evaluate_model(model_bn, "BatchNorm")[0])

fig, axes = plt.subplots(1, 2, figsize=(13,4))
for label, h in [("Baseline", histories["Baseline"]), ("BatchNorm", history_bn)]:
    axes[0].plot(h.history["loss"], label=f"{label} train")
    axes[0].plot(h.history["val_loss"], linestyle="--", label=f"{label} val")
    axes[1].plot(h.history["accuracy"], label=f"{label} train")
    axes[1].plot(h.history["val_accuracy"], linestyle="--", label=f"{label} val")
axes[0].set_title("Loss"); axes[1].set_title("Accuracy")
for ax in axes:
    ax.set_xlabel("Epoch"); ax.legend()
plt.suptitle("Batch Normalization vs Baseline — Training Dynamics")
plt.tight_layout(); plt.show()

**Batch normalization:** Normalizes intermediate activations using mini-batch statistics during training and moving statistics during inference, then applies learnable scale (`gamma`) and shift (`beta`). The common ordering is Dense → BatchNormalization → Activation. It may improve optimization stability, but the actual benefit should be judged from this experiment.

In [ ]:
# Compare BN before activation (canonical) with BN after activation.
def build_bn_position(input_dim, bn_after_activation=False):
    inp = tf.keras.layers.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(128, kernel_initializer="he_normal")(inp)
    if bn_after_activation:
        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.BatchNormalization()(x)
    else:
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.Dense(64, kernel_initializer="he_normal")(x)
    if bn_after_activation:
        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.BatchNormalization()(x)
    else:
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
    out = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    m = tf.keras.Model(inp, out)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

model_bn_after = build_bn_position(X_fit.shape[1], bn_after_activation=True)
model_bn_before = build_bn_position(X_fit.shape[1], bn_after_activation=False)
hist_bn_after = train_model(model_bn_after, epochs=30, verbose=0)
hist_bn_before = train_model(model_bn_before, epochs=30, verbose=0)
plt.figure(figsize=(9,4))
plt.plot(hist_bn_after.history["val_accuracy"], label="Activation → BN")
plt.plot(hist_bn_before.history["val_accuracy"], label="BN → Activation")
plt.title("BatchNorm Position Experiment")
plt.xlabel("Epoch"); plt.ylabel("Validation accuracy"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Inspect gamma and beta of the first BN layer.
bn_layer = next(layer for layer in model_bn.layers if isinstance(layer, tf.keras.layers.BatchNormalization))
bn_weights = bn_layer.get_weights()
gamma, beta = bn_weights[0], bn_weights[1]
fig, axes = plt.subplots(2, 1, figsize=(13,6))
axes[0].bar(np.arange(len(gamma)), gamma)
axes[0].set_title("BatchNorm Learned Gamma"); axes[0].set_xlabel("Neuron"); axes[0].set_ylabel("Gamma")
axes[1].bar(np.arange(len(beta)), beta)
axes[1].set_title("BatchNorm Learned Beta"); axes[1].set_xlabel("Neuron"); axes[1].set_ylabel("Beta")
plt.tight_layout(); plt.show()

## Task 7 — Optimizers, Learning Rate Sensitivity & Final Comparison
**Objective:** Compare optimizers and learning rates, train a combined model, and summarize all model metrics.

**Expected output:** Optimizer convergence plot, learning-rate plots, final model evaluation, results table, and ROC curves.

In [ ]:
optimizer_configs = {
    "SGD": "sgd",
    "SGD + Momentum": tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSprop": "rmsprop",
    "Adam": "adam",
    "Adam explicit": tf.keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)
}
optimizer_models, optimizer_histories = {}, {}
for name, opt in optimizer_configs.items():
    print("Training optimizer:", name)
    m = build_ann(input_dim=X_fit.shape[1], activation="relu", initializer="he_normal",
                  use_batch_norm=True, optimizer=opt, loss="binary_crossentropy")
    h = train_model(m, epochs=50, verbose=0)
    optimizer_models[name], optimizer_histories[name] = m, h
    models["Optimizer: " + name] = m
    results.append(evaluate_model(m, "Optimizer: " + name)[0])

plt.figure(figsize=(12,6))
for name, h in optimizer_histories.items():
    plt.plot(h.history["val_accuracy"], label=name)
plt.axhline(0.80, linestyle="--", color="gray", label="80% reference")
plt.axhline(0.86, linestyle="--", color="black", label="86% reference")
plt.title("Optimizer Comparison — Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Validation accuracy"); plt.legend()
plt.tight_layout(); plt.show()

**Optimizer summary:** Vanilla SGD uses one global learning rate. Momentum smooths updates and can speed progress through shallow regions. RMSprop adapts the learning rate using a running average of squared gradients. Adam combines first-moment momentum and second-moment scaling with bias correction, making it a strong default for sparse, high-dimensional one-hot encoded features. Actual ranking depends on the run and dataset.

In [ ]:
# Learning-rate sensitivity: 4 SGD rates and 3 Adam rates.
lr_histories = {"SGD": {}, "Adam": {}}
for lr in [0.0001, 0.001, 0.01, 0.1]:
    opt = tf.keras.optimizers.SGD(learning_rate=lr)
    m = build_ann(input_dim=X_fit.shape[1], activation="relu", initializer="he_normal",
                  use_batch_norm=True, optimizer=opt)
    lr_histories["SGD"][lr] = train_model(m, epochs=30, verbose=0)
for lr in [0.0001, 0.001, 0.01]:
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    m = build_ann(input_dim=X_fit.shape[1], activation="relu", initializer="he_normal",
                  use_batch_norm=True, optimizer=opt)
    lr_histories["Adam"][lr] = train_model(m, epochs=30, verbose=0)

fig, axes = plt.subplots(1,2, figsize=(14,5))
for lr, h in lr_histories["SGD"].items():
    axes[0].plot(h.history["val_loss"], label=f"lr={lr}")
for lr, h in lr_histories["Adam"].items():
    axes[1].plot(h.history["val_loss"], label=f"lr={lr}")
axes[0].set_title("SGD Learning Rate Sensitivity"); axes[1].set_title("Adam Learning Rate Sensitivity")
for ax in axes:
    ax.set_xlabel("Epoch"); ax.set_ylabel("Validation loss"); ax.legend()
plt.tight_layout(); plt.show()

**Final model:** The following combined model uses ReLU, He Normal initialization, batch normalization, and Adam with BCE. The notebook also calculates a weighted-BCE alternative; compare class-1 F1 and recall before selecting a production candidate. The held-out test set should be used for final reporting, not for repeatedly tuning the model.

In [ ]:
# Select a final configuration. Weighted BCE is included because class imbalance is material.
final_model = build_ann(input_dim=X_fit.shape[1], hidden_units=(128,64), activation="relu",
                        initializer="he_normal", use_batch_norm=True,
                        optimizer="adam", loss="binary_crossentropy")
final_history = train_model(final_model, epochs=80, class_weight=class_weight_dict, verbose=0)
models["Final Combined"] = final_model
final_result, final_prob, final_pred = evaluate_model(final_model, "Final Combined")
results.append(final_result)
print(classification_report(y_test, final_pred, target_names=["<=50K (0)", ">50K (1)"], zero_division=0))

In [ ]:
# Full comparison table. De-duplicate names by keeping the last recorded row for each model name.
results_df = pd.DataFrame(results).drop_duplicates(subset=["Model"], keep="last")
results_df = results_df[["Model", "Test Acc", "Prec(1)", "Recall(1)", "F1(1)", "ROC-AUC"]]
display(results_df.style.highlight_max(subset=["F1(1)", "ROC-AUC"], color="#ffcccc").format({
    "Test Acc":"{:.3f}", "Prec(1)":"{:.3f}", "Recall(1)":"{:.3f}", "F1(1)":"{:.3f}", "ROC-AUC":"{:.3f}"
}))

In [ ]:
# ROC curves: baseline, weighted BCE, and final combined model.
plt.figure(figsize=(8,6))
for label, model in [("Baseline", model_base), ("Weighted BCE", model_wbce), ("Final Combined", final_model)]:
    prob = model.predict(X_test, verbose=0).ravel()
    RocCurveDisplay.from_predictions(y_test, prob, name=f"{label} (AUC={roc_auc_score(y_test, prob):.3f})")
plt.plot([0,1], [0,1], linestyle="--", color="gray")
plt.title("ROC Curves — Baseline, Weighted BCE, Final Combined")
plt.tight_layout(); plt.show()

## Final analysis and conclusion

Complete these points after running the notebook:

1. **Best activation:** identify the activation with the highest validation class-1 F1 and discuss its accuracy/F1 trade-off.
2. **Best initializer:** compare convergence speed and final class-1 F1. Explain the zero-initialization symmetry problem.
3. **Best loss:** compare BCE, MSE, weighted BCE, and focal loss. Explain any precision/recall trade-off.
4. **Batch normalization:** compare training stability and held-out metrics; mention which BN placement worked better in this run.
5. **Best optimizer:** report which optimizer converged fastest and which achieved the highest class-1 F1.
6. **Final model:** report test accuracy, class-1 precision, recall, F1, and ROC-AUC from `results_df`.
7. **Limitations:** the dataset has class imbalance and demographic features. Predictive associations do not imply causation, and income predictions can encode or amplify social bias. Do not use this model for consequential decisions without fairness, privacy, and deployment review.

## Reproducibility checklist

- [ ] `adult.csv` or `adult.csv.zip` is beside this notebook.
- [ ] Run **Kernel → Restart Kernel and Run All**.
- [ ] Confirm all plots and the results table render without errors.
- [ ] Save the executed notebook as `DL_PR2.ipynb`.
- [ ] Optionally export an HTML copy and include `requirements.txt` in your repository.